In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

from clip_modules.modeling.model import CLIPRModel


def get_args_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--nb_knn", nargs="+", type=int, default=[10])
    parser.add_argument("--temperature", type=float, default=0.05)
    parser.add_argument("--gather-on-cpu", action="store_true")
    parser.add_argument("--architecture", default="DinoV2")
    parser.add_argument("--config_file", type=str, default="dinov2/configs/train/vitl16.yaml")
    parser.add_argument("--opts", default=[], nargs="+")
    parser.add_argument("--output_dir", default="ModelSavedOnlyKNN_DinoCLIP_439399_Total")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--n-per-class-list", nargs="+", type=int, default=[-1])
    parser.add_argument("--n-tries", type=int, default=1)
    return parser


def create_model():
    args = get_args_parser().parse_args([])

    weight_path = "./RetiGlobe.pth"

    DinoCLIP = CLIPRModel(
        vision_type="DinoV2",
        from_checkpoint=True,
        vision_pretrained=True,
        weights_path=weight_path,
        R=8,
        args=args
    )

    model = DinoCLIP.vision_model.model
    model = model.cuda()
    return model


class MLPClassifier(nn.Module):
    def __init__(self, in_dim=1024, hidden1=512, hidden2=256, hidden3=128, num_classes=2):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, hidden3),
            nn.ReLU()
        )

        self.classifier = nn.Linear(hidden3, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)
        features = F.normalize(features, dim=1)
        logits = self.classifier(features)
        return logits, features


def load_image(path, transform):
    img = Image.open(path).convert("RGB")
    img = transform(img)
    img = img.unsqueeze(0)   # [1, 3, 224, 224]
    return img.cuda()


if __name__ == "__main__":

    left_img_path = "/data4/XY/Code_Ocean/test_image/1_L.png"
    right_img_path = "/data4/XY/Code_Ocean/test_image/1_R.png"

    best_model_path = "paired_ssl_best_model_20_new.pth"

    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.3741, 0.2282, 0.1176],
            std=[0.2557, 0.1542, 0.0814]
        )
    ])

    print("Loading backbone...")
    model = create_model()

    print("Loading checkpoint...")
    checkpoint = torch.load(best_model_path, map_location="cuda", weights_only=False)

    model.load_state_dict(checkpoint["backbone"])
    model.eval()

    mlp = MLPClassifier().cuda()
    mlp.load_state_dict(checkpoint["mlp"])
    mlp.eval()

    print("Loading image pair...")
    img_L = load_image(left_img_path, transform_val)
    img_R = load_image(right_img_path, transform_val)

    print("Image L shape:", img_L.shape)
    print("Image R shape:", img_R.shape)

    with torch.no_grad():
        feat_L = model(img_L)
        feat_R = model(img_R)

        logits_L, emb_L = mlp(feat_L)
        logits_R, emb_R = mlp(feat_R)

        prob_L = F.softmax(logits_L, dim=1)[:, 1]
        prob_R = F.softmax(logits_R, dim=1)[:, 1]

        avg_prob = (prob_L + prob_R) / 2
        pred = (avg_prob >= 0.5).long()

    print("\n===== Single Pair Output =====")
    print("Left logits: ", logits_L.cpu().numpy())
    print("Right logits:", logits_R.cpu().numpy())

    print("Left positive prob: ", prob_L.item())
    print("Right positive prob:", prob_R.item())

    print("Average positive prob:", avg_prob.item())
    print("Prediction:", pred.item())

Loading backbone...
load model weight from: ./RetiGlobe.pth
Loading checkpoint...
Loading image pair...
Image L shape: torch.Size([1, 3, 224, 224])
Image R shape: torch.Size([1, 3, 224, 224])

===== Single Pair Output =====
Left logits:  [[-0.4178433   0.33910683]]
Right logits: [[-0.34003127  0.24492747]]
Left positive prob:  0.6806912422180176
Right positive prob: 0.6422075629234314
Average positive prob: 0.6614494323730469
Prediction: 1
